# 10. MoE and routing — DeepSeek-V4 and Kimi-K3 faithful micro-implementations

Only width/expert counts are reduced. The routing, expert, latent-MoE, balancing and MTP paths are kept.

- DeepSeekMoE shared + routed experts
- V4 `sqrt(softplus)` affinity, hash routing, no-aux correction bias, clamped SwiGLU
- K3 Stable LatentMoE + SiTU-GLU + exact Quantile Balancing update
- Multi-Token Prediction training heads


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 1. SwiGLU expert and DeepSeek-V4 clamping


In [ ]:
class SwiGLUExpert(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim=None, clamp_limit=None):
        super().__init__()
        output_dim = input_dim if output_dim is None else output_dim
        self.clamp_limit = clamp_limit
        self.gate = nn.Linear(input_dim, hidden_dim, bias=False)
        self.value = nn.Linear(input_dim, hidden_dim, bias=False)
        self.out = nn.Linear(hidden_dim, output_dim, bias=False)

    def forward(self, x):
        gate = self.gate(x)
        value = self.value(x)
        if self.clamp_limit is not None:
            gate = gate.clamp(max=self.clamp_limit)
            value = value.clamp(-self.clamp_limit, self.clamp_limit)
        return self.out(F.silu(gate) * value)


## 2. DeepSeek-V4 gate: sqrt-softplus, correction bias, and hash routing

The correction bias changes expert selection only. Mixture weights are gathered from the unbiased affinities. In hash mode, token IDs choose the experts while learned affinities still determine their weights.


In [ ]:
class V4Gate(nn.Module):
    def __init__(self, model_dim=24, experts=8, top_k=2, vocab=64, hash_routing=False):
        super().__init__()
        self.experts = experts
        self.top_k = top_k
        self.hash_routing = hash_routing
        self.weight = nn.Parameter(torch.randn(experts, model_dim) * 0.02)
        self.correction_bias = nn.Parameter(torch.zeros(experts), requires_grad=False)

        table = torch.empty(vocab, top_k, dtype=torch.long)
        for token_id in range(vocab):
            table[token_id] = torch.tensor([(token_id + j) % experts for j in range(top_k)])
        self.register_buffer("token_to_expert", table)

    def forward(self, x, token_ids=None):
        logits = F.linear(x.float(), self.weight.float())
        affinity = torch.sqrt(F.softplus(logits))

        if self.hash_routing:
            if token_ids is None:
                raise ValueError("token_ids are required for hash routing")
            ids = self.token_to_expert[token_ids]
        else:
            ids = (affinity + self.correction_bias).topk(self.top_k, dim=-1).indices

        weights = affinity.gather(-1, ids)
        weights = weights / weights.sum(dim=-1, keepdim=True).clamp_min(1e-8)
        return weights.to(x.dtype), ids, affinity.to(x.dtype)


## 3. V4 routed/shared MoE layer


In [ ]:
class TinyV4MoE(nn.Module):
    def __init__(self, model_dim=24, experts=8, top_k=2, hash_routing=False):
        super().__init__()
        self.top_k = top_k
        self.gate = V4Gate(model_dim, experts, top_k, hash_routing=hash_routing)
        self.routed = nn.ModuleList([SwiGLUExpert(model_dim, 32, clamp_limit=10.0) for _ in range(experts)])
        self.shared = SwiGLUExpert(model_dim, 32, clamp_limit=10.0)

    def forward(self, tokens, token_ids=None):
        flat = tokens.reshape(-1, tokens.size(-1))
        flat_ids = None if token_ids is None else token_ids.reshape(-1)
        weights, ids, affinity = self.gate(flat, flat_ids)
        routed = torch.zeros_like(flat)

        for slot in range(self.top_k):
            slot_ids = ids[:, slot]
            slot_weights = weights[:, slot]
            for expert_id, expert in enumerate(self.routed):
                mask = slot_ids == expert_id
                if mask.any():
                    routed[mask] += slot_weights[mask, None] * expert(flat[mask])

        output = flat + self.shared(flat) + routed
        return output.view_as(tokens), {"ids": ids, "weights": weights, "affinity": affinity}


tokens = torch.randn(2, 6, 24, device=device)
token_ids = torch.randint(0, 64, (2, 6), device=device)

hash_moe = TinyV4MoE(hash_routing=True).to(device)
learned_moe = TinyV4MoE(hash_routing=False).to(device)

hash_output, hash_diag = hash_moe(tokens, token_ids)
learned_output, learned_diag = learned_moe(tokens)
print("hash ids:", hash_diag["ids"][:4])
print("learned ids:", learned_diag["ids"][:4])
print("V4 MoE output:", learned_output.shape)


## 4. SiTU-GLU for K3


In [ ]:
class SiTUGLU(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, beta_gate=4.0, beta_value=25.0):
        super().__init__()
        self.beta_gate = beta_gate
        self.beta_value = beta_value
        self.gate = nn.Linear(input_dim, hidden_dim, bias=False)
        self.value = nn.Linear(input_dim, hidden_dim, bias=False)
        self.out = nn.Linear(hidden_dim, output_dim, bias=False)

    def forward(self, x):
        gate_pre = self.gate(x)
        value_pre = self.value(x)
        gate = self.beta_gate * torch.tanh(gate_pre / self.beta_gate) * torch.sigmoid(gate_pre)
        value = self.beta_value * torch.tanh(value_pre / self.beta_value)
        return self.out(gate * value)


## 5. Exact Quantile Balancing bias update

Let `alpha_i` be the `(k+1)`-th largest biased score for token `i`. The next-step routing bias for expert `j` is the negative `(1-k/n)` quantile of the margin `s_ij-alpha_i`, then mean-centered. The new bias is used on the next routing step; it never directly scales expert outputs.


In [ ]:
@torch.no_grad()
def quantile_balance_update(router_scores, routing_bias, top_k):
    expert_count = router_scores.size(-1)
    biased_scores = router_scores + routing_bias
    top_k_plus_one = biased_scores.topk(top_k + 1, dim=-1).values
    cutoff = top_k_plus_one[:, top_k]

    margins = router_scores - cutoff[:, None]
    target_quantile = 1.0 - top_k / expert_count
    next_bias = -torch.quantile(margins, target_quantile, dim=0)
    next_bias = next_bias - next_bias.mean()
    routing_bias.copy_(next_bias)

    selected = biased_scores.topk(top_k, dim=-1).indices
    load = torch.stack([(selected == j).float().mean() for j in range(expert_count)])
    return {"load": load, "cutoff": cutoff, "next_bias": next_bias}


## 6. Stable LatentMoE

Only routed experts operate in the smaller latent width. The routed aggregate is normalized before up-projection; shared experts remain full width.


In [ ]:
class TinyStableLatentMoE(nn.Module):
    def __init__(self, model_dim=24, latent_dim=12, experts=8, top_k=2):
        super().__init__()
        self.top_k = top_k
        self.experts = experts
        self.down = nn.Linear(model_dim, latent_dim, bias=False)
        self.latent_norm = nn.RMSNorm(latent_dim)
        self.up = nn.Linear(latent_dim, model_dim, bias=False)
        self.router = nn.Linear(model_dim, experts, bias=False)
        self.routing_bias = nn.Parameter(torch.zeros(experts), requires_grad=False)
        self.routed = nn.ModuleList([SiTUGLU(latent_dim, 24, latent_dim) for _ in range(experts)])
        self.shared = SiTUGLU(model_dim, 48, model_dim)

    def forward(self, tokens):
        latent = self.down(tokens)
        raw_scores = torch.sigmoid(self.router(tokens))
        ids = (raw_scores + self.routing_bias).topk(self.top_k, dim=-1).indices
        selected_raw = raw_scores.gather(-1, ids)
        weights = selected_raw / selected_raw.sum(dim=-1, keepdim=True).clamp_min(1e-8)

        routed_output = torch.zeros_like(latent)
        flat_latent = latent.reshape(-1, latent.size(-1))
        flat_ids = ids.reshape(-1, self.top_k)
        flat_weights = weights.reshape(-1, self.top_k)
        flat_output = routed_output.reshape(-1, latent.size(-1))

        for slot in range(self.top_k):
            for expert_id, expert in enumerate(self.routed):
                mask = flat_ids[:, slot] == expert_id
                if mask.any():
                    flat_output[mask] += flat_weights[mask, slot, None] * expert(flat_latent[mask])

        routed_full = self.up(self.latent_norm(routed_output))
        return tokens + routed_full + self.shared(tokens), {"raw_scores": raw_scores, "ids": ids, "weights": weights}


k3_moe = TinyStableLatentMoE().to(device)
k3_tokens = torch.randn(2, 8, 24, device=device)
k3_output, k3_diag = k3_moe(k3_tokens)
flat_scores = k3_diag["raw_scores"].detach().reshape(-1, k3_moe.experts)
balance = quantile_balance_update(flat_scores, k3_moe.routing_bias, k3_moe.top_k)
loss = k3_output.square().mean()
loss.backward()
print("Stable LatentMoE:", k3_output.shape)
print("mixture sums:", k3_diag["weights"].sum(dim=-1)[0])
print("next routing bias:", balance["next_bias"])
print("down-proj grad:", k3_moe.down.weight.grad.norm().item())


## 7. Multi-Token Prediction

V4 retains the V3 MTP training strategy, so several future offsets are predicted from the same causal backbone instead of omitting this training path.


In [ ]:
class TinyMTPModel(nn.Module):
    def __init__(self, vocab=64, model_dim=24, offsets=(1, 2)):
        super().__init__()
        self.offsets = offsets
        self.embedding = nn.Embedding(vocab, model_dim)
        layer = nn.TransformerEncoderLayer(model_dim, 3, 4 * model_dim, batch_first=True)
        self.backbone = nn.TransformerEncoder(layer, num_layers=2)
        self.norm = nn.RMSNorm(model_dim)
        self.heads = nn.ModuleList([nn.Linear(model_dim, vocab, bias=False) for _ in offsets])

    def forward(self, token_ids):
        length = token_ids.size(1)
        mask = torch.triu(torch.ones(length, length, dtype=torch.bool, device=token_ids.device), diagonal=1)
        hidden = self.norm(self.backbone(self.embedding(token_ids), mask=mask))
        return [head(hidden) for head in self.heads]


mtp = TinyMTPModel().to(device)
ids = torch.randint(0, 64, (3, 10), device=device)
logits_list = mtp(ids)
mtp_loss = torch.zeros((), device=device)
for offset, logits in zip(mtp.offsets, logits_list):
    mtp_loss = mtp_loss + F.cross_entropy(logits[:, :-offset].reshape(-1, 64), ids[:, offset:].reshape(-1))
mtp_loss.backward()
print("MTP loss:", mtp_loss.item())


## References and provenance

- DeepSeek-V4 public implementation: `sqrt(softplus)` affinity, hash-routed early MoE layers, correction bias that affects selection only, normalized unbiased mixture weights, clamped SwiGLU, shared expert.
- Kimi-K3: Stable LatentMoE, SiTU-GLU and Quantile Balancing.
- DeepSeek-V4 model card: V4 retains Multi-Token Prediction from V3.
